# 🔍 Week 6: Named Entity Recognition (NER)

## Tujuan Pembelajaran:
1. Memahami konsep **Named Entity Recognition (NER)** — mengidentifikasi dan mengklasifikasikan entitas bernama (nama orang, lokasi, organisasi, dll.) dalam teks
2. Menggunakan model **spaCy pretrained** (`en_core_web_sm`) untuk NER pada teks Inggris
3. Mengenal jenis-jenis entitas dalam skema **OntoNotes** yang digunakan spaCy
4. Menerapkan **EntityRuler** untuk menambahkan aturan NER berbasis pola (phrase & token matching)
5. Melakukan NER pada **teks bahasa Indonesia** menggunakan model **BERT** (`cahya/bert-base-indonesian-NER`)
6. Menerapkan NER ke seluruh dataset ulasan Honest Review

---
> 📂 **Dataset**: Honest Review (`cleandata.csv`) — kolom `text_final` (teks yang sudah dipreproses)
> File ini berada di folder `Week 3/` dalam workspace yang sama.


## 1) Install Library

Install spaCy, model bahasa Inggris, dan library Transformers untuk model NER bahasa Indonesia.

In [1]:
%pip install spacy transformers sentencepiece pandas openpyxl

  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/14.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/14.4 MB ? eta -:--:--
    --------------------------------------- 0.3/14.4 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.4 MB 798.3 kB/s eta 0:00:18
   - --------------------

In [3]:
# Download model spaCy bahasa Inggris
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - ------------------------------------- 0.5/12.8 MB 304.5 kB/s eta 0:00:41
     - ------------------------------------- 0.5/12.8 MB 304.5 kB/s eta 0:00:41
     - ------------------------------------- 0.5/12.8 MB 304.5 kB/s eta 0:00:41
     -- ------------------------------------ 0.8/12.8 MB 392.4 kB/s eta 0:00:31
     -- ------------------------------------ 0.8/12.8 MB 392.4 kB/s eta 0:00:31
     --- ----------------------------------- 1.0/12.8 MB 418.3 kB/s eta 0:00:29
     --- -------

## 2) NER dengan spaCy — Model Pretrained

spaCy menyediakan model NER yang sudah dilatih pada corpus besar. Model `en_core_web_sm` dapat mengenali entitas seperti nama orang, organisasi, lokasi, tanggal, dan lainnya dari teks Inggris.

Pipeline spaCy terdiri dari:
1. `tok2vec` — representasi token
2. `tagger` — POS tagging
3. `parser` — dependency parsing
4. `ner` — named entity recognition

In [4]:
import spacy
from spacy import displacy

# Load model spaCy bahasa Inggris
nlp = spacy.load('en_core_web_sm')
print("Model spaCy berhasil dimuat.")
print(f"Pipeline: {nlp.pipe_names}")

Model spaCy berhasil dimuat.
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [5]:
# Contoh NER pada teks Inggris
text = (
    "George Washington was the first president of the United States. "
    "He was born on February 22, 1732, in Virginia. "
    "He led the Continental Army and later served two terms as president."
)

doc = nlp(text)

print("Entitas yang ditemukan:")
print(f"{'Entitas':<30} {'Label':<12} {'Keterangan'}")
print('-' * 65)
for ent in doc.ents:
    print(f"{ent.text:<30} {ent.label_:<12} {spacy.explain(ent.label_)}")

Entitas yang ditemukan:
Entitas                        Label        Keterangan
-----------------------------------------------------------------
George Washington              PERSON       People, including fictional
first                          ORDINAL      "first", "second", etc.
the United States              GPE          Countries, cities, states
February 22, 1732              DATE         Absolute or relative dates or periods
Virginia                       GPE          Countries, cities, states
the Continental Army           ORG          Companies, agencies, institutions, etc.
two                            CARDINAL     Numerals that do not fall under another type


In [6]:
# Visualisasi NER dengan displacy
displacy.render(doc, style='ent', jupyter=True)

## 3) Jenis Entitas OntoNotes (spaCy)

spaCy menggunakan skema entity **OntoNotes 5**. Berikut daftar lengkap entitas yang didukung:

| Label      | Keterangan |
|------------|------------|
| `PERSON`   | Nama orang, termasuk tokoh fiksi |
| `NORP`     | Kebangsaan, kelompok agama atau politik |
| `FAC`      | Fasilitas: bangunan, bandara, jembatan, jalan |
| `ORG`      | Perusahaan, lembaga, instansi pemerintah |
| `GPE`      | Negara, kota, provinsi |
| `LOC`      | Lokasi non-GPE: gunung, lautan, sungai |
| `PRODUCT`  | Objek, kendaraan, makanan |
| `EVENT`    | Perang, pertempuran, olahraga, dll. |
| `WORK_OF_ART` | Judul buku, lagu, karya seni |
| `LAW`      | Peraturan, undang-undang |
| `LANGUAGE` | Bahasa yang disebutkan |
| `DATE`     | Tanggal atau periode waktu |
| `TIME`     | Waktu lebih kecil dari sehari |
| `PERCENT`  | Persentase |
| `MONEY`    | Nilai moneter |
| `QUANTITY` | Ukuran, berat, jarak |
| `ORDINAL`  | "pertama", "kedua", dll. |
| `CARDINAL` | Angka yang tidak termasuk kategori lain |

In [7]:
# Cetak semua entitas yang didukung model
print("Entitas yang dikenali oleh model en_core_web_sm:")
for label in nlp.get_pipe('ner').labels:
    print(f"  {label:<15} → {spacy.explain(label)}")

Entitas yang dikenali oleh model en_core_web_sm:
  CARDINAL        → Numerals that do not fall under another type
  DATE            → Absolute or relative dates or periods
  EVENT           → Named hurricanes, battles, wars, sports events, etc.
  FAC             → Buildings, airports, highways, bridges, etc.
  GPE             → Countries, cities, states
  LANGUAGE        → Any named language
  LAW             → Named documents made into laws.
  LOC             → Non-GPE locations, mountain ranges, bodies of water
  MONEY           → Monetary values, including unit
  NORP            → Nationalities or religious or political groups
  ORDINAL         → "first", "second", etc.
  ORG             → Companies, agencies, institutions, etc.
  PERCENT         → Percentage, including "%"
  PERSON          → People, including fictional
  PRODUCT         → Objects, vehicles, foods, etc. (not services)
  QUANTITY        → Measurements, as of weight or distance
  TIME            → Times smaller than 

## 4) EntityRuler — Phrase Matching (Aturan Berbasis Teks Eksak)

**EntityRuler** memungkinkan kita mendefinisikan aturan NER berbasis pola, bukan statistik. Berguna ketika:
- Ada daftar nama yang pasti (misalnya: nama presiden, nama kota)
- Entitas memiliki pola yang sangat terstruktur (misalnya: nomor IP, kode produk)

**Phrase patterns** cocok untuk pencocokan string eksak.

In [8]:
# Tambahkan EntityRuler ke pipeline — SEBELUM komponen 'ner'
# Ini penting agar 'ner' menghormati label dari EntityRuler
ruler = nlp.add_pipe("entity_ruler", before="ner", name="presidents")

# Daftar nama presiden AS
presidents = [
    "George Washington", "John Adams", "Thomas Jefferson", "Abraham Lincoln",
    "Franklin D. Roosevelt", "Harry S. Truman", "Dwight D. Eisenhower",
    "John F. Kennedy", "Lyndon B. Johnson", "Richard Nixon",
    "Jimmy Carter", "Ronald Reagan", "George H.W. Bush",
    "Bill Clinton", "George W. Bush", "Barack Obama", "Donald Trump"
]

def make_patterns(name_list, label):
    """Ubah list nama menjadi format pola EntityRuler."""
    return [{"label": label, "pattern": name} for name in name_list]

# Tambahkan pola ke EntityRuler
ruler.add_patterns(make_patterns(presidents, "PRES"))

print(f"EntityRuler ditambahkan. Pipeline: {nlp.pipe_names}")
print(f"Jumlah pola terdaftar: {len(ruler.patterns)}")

EntityRuler ditambahkan. Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'presidents', 'ner']
Jumlah pola terdaftar: 17


In [9]:
# Test EntityRuler
text2 = (
    "Barack Obama and George W. Bush both attended the inauguration of Donald Trump "
    "at the Capitol in Washington D.C. in January 2017."
)

doc2 = nlp(text2)

print("Entitas yang ditemukan (dengan EntityRuler):")
print(f"{'Entitas':<30} {'Label'}")
print('-' * 45)
for ent in doc2.ents:
    print(f"{ent.text:<30} {ent.label_}")

print()
displacy.render(doc2, style='ent', jupyter=True)

Entitas yang ditemukan (dengan EntityRuler):
Entitas                        Label
---------------------------------------------
Barack Obama                   PRES
George W. Bush                 PRES
Donald Trump                   PRES
Capitol                        FAC
Washington D.C.                GPE
January 2017                   DATE



## 5) EntityRuler — Token Patterns (Aturan Berbasis Atribut Token)

**Token patterns** lebih fleksibel dari phrase patterns — bisa menggunakan regex dan atribut seperti `LOWER`, `POS`, `SHAPE`.

Contoh: mencocokkan berbagai cara penulisan nama "George H.W. Bush":
- 2 token: `george bush`
- 3 token: `george hw bush`, `george h.w. bush`
- 4 token: `george h. w. bush`, `george herbert walker bush`

In [10]:
# Hapus EntityRuler lama dan buat ulang dengan token patterns
nlp.remove_pipe('presidents')
ruler2 = nlp.add_pipe("entity_ruler", before="ner", name="presidents")

token_patterns = [
    # 2 token: "george bush"
    {"id": "PRES41", "label": "PRES",
     "pattern": [{"LOWER": "george"}, {"LOWER": "bush"}]},

    # 3 token: "george hw bush", "george h.w. bush"
    {"id": "PRES41", "label": "PRES",
     "pattern": [{"LOWER": "george"}, {"LOWER": {"REGEX": "h(\\.w\\.?|w)"}}, {"LOWER": "bush"}]},

    # 4 token: "george herbert walker bush"
    {"id": "PRES41", "label": "PRES",
     "pattern": [{"LOWER": "george"}, {"LOWER": {"REGEX": "h(erbert|\\.?)"}},
                 {"LOWER": {"REGEX": "w(alker|\\.?)"}}, {"LOWER": "bush"}]},
]

ruler2.add_patterns(token_patterns)

# Uji berbagai format nama
text3 = (
    "Known names: George Bush, George HW Bush, George H.W. Bush, "
    "George Herbert Walker Bush. His son George W. Bush was also president."
)
doc3 = nlp(text3)

print("Entitas yang ditemukan (Token Patterns):")
for ent in doc3.ents:
    print(f"  {ent.text!r:<40} → {ent.label_}")

displacy.render(doc3, style='ent', jupyter=True)

Entitas yang ditemukan (Token Patterns):
  'George Bush'                            → PRES
  'George HW Bush'                         → PRES
  'George H.W. Bush'                       → PRES
  'George Herbert Walker Bush'             → PRES
  'George W. Bush'                         → PERSON


## 6) NER Bahasa Indonesia dengan BERT

Untuk teks bahasa Indonesia, kita menggunakan model **`cahya/bert-base-indonesian-NER`** — model BERT yang dilatih khusus untuk NER pada teks Indonesia. Model ini mengenali entitas:

| Label | Keterangan |
|-------|------------|
| `PER` | Nama orang (Person) |
| `ORG` | Organisasi |
| `LOC` | Lokasi |
| `QTY` | Kuantitas |
| `EVT` | Event/Kejadian |
| `PRD` | Produk |
| `TIM` | Waktu |

In [11]:
from transformers import pipeline, AutoTokenizer

# Load tokenizer dan model NER bahasa Indonesia
print("Loading model NER bahasa Indonesia (cahya/bert-base-indonesian-NER)...")
tokenizer = AutoTokenizer.from_pretrained("cahya/bert-base-indonesian-NER")
ner_pipeline_id = pipeline(
    "ner",
    model="cahya/bert-base-indonesian-NER",
    tokenizer=tokenizer,
    aggregation_strategy="simple"  # Gabungkan sub-token menjadi satu entitas
)
print("Model berhasil dimuat!")

c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model NER bahasa Indonesia (cahya/bert-base-indonesian-NER)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30135.63it/s]
BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model berhasil dimuat!


In [12]:
# Uji NER pada kalimat bahasa Indonesia
teks_id = "Jokowi mengunjungi Bandung untuk meresmikan proyek infrastruktur baru bersama Menteri PUPR Basuki."

results = ner_pipeline_id(teks_id)

print("=== Hasil NER Bahasa Indonesia ===")
print(f"Teks: {teks_id}\n")
print(f"{'Token':<25} {'Entitas':<10} {'Score'}")
print('-' * 50)
for r in results:
    print(f"{r['word']:<25} {r['entity_group']:<10} {r['score']:.4f}")

=== Hasil NER Bahasa Indonesia ===
Teks: Jokowi mengunjungi Bandung untuk meresmikan proyek infrastruktur baru bersama Menteri PUPR Basuki.

Token                     Entitas    Score
--------------------------------------------------
jokowi                    PER        0.9877
bandung                   GPE        0.9920
menteri pupr              NOR        0.9467
basuki                    PER        0.9905


In [ ]:
# Contoh lain: teks berkaitan dengan ulasan produk keuangan Indonesia
teks_id2 = (
    "BCA dan Mandiri bekerja sama dengan OJK untuk meningkatkan "
    "layanan perbankan digital di Jakarta dan Surabaya pada tahun 2024."
)

results_id2 = ner_pipeline_id(teks_id2)

print("=== NER Teks Ulasan Keuangan ===")
print(f"Teks: {teks_id2}\n")
print(f"{'Token':<30} {'Entitas':<10} {'Score'}")
print('-' * 55)
for r in results_id2:
    print(f"{r['word']:<30} {r['entity_group']:<10} {r['score']:.4f}")


=== NER Teks BPJS ===
Teks: BPJS Kesehatan bekerja sama dengan Kementerian Kesehatan untuk meningkatkan kualitas layanan JKN di Jakarta dan Surabaya pada tahun 2023.

Token                          Entitas    Score
-------------------------------------------------------
bpjs kesehatan                 NOR        0.8068
kementerian kesehatan          NOR        0.9829
##n                            PRD        0.4112
jakarta                        GPE        0.9974
surabaya                       GPE        0.9815
2023                           DAT        0.9162


## 7) Terapkan NER ke Seluruh Dataset Honest Review

Load dataset `cleandata.csv` dan terapkan model NER bahasa Indonesia ke kolom `text_final`. Hasil entitas disimpan sebagai kolom baru `NER_Results` dalam format string.


In [15]:
import pandas as pd
import unicodedata
import re

# Load dataset Honest Review
file_path = '../Week 3/cleandata.csv'
df_text = pd.read_csv(file_path)
print(f"Jumlah data: {len(df_text)}")
print(f"Kolom: {list(df_text.columns)}")
df_text['text_final'].head()


Jumlah data: 39164
Kolom: ['score', 'content', 'text_final', 'reviewCreatedVersion']


0    kali kartu kredit cocok anak muda update alam ...
1    hasil layan cs nama sangat sopan jelas sangat ...
2                                           cepat aman
3                                              cs baik
4                                           jelas baik
Name: text_final, dtype: str

In [16]:
def clean_text(text):
    """Bersihkan teks dari karakter non-printable dan non-ASCII."""
    text = str(text)
    text = ''.join(char for char in text if unicodedata.category(char)[0] != 'C')
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text


def perform_ner(text, max_length=512):
    """
    Terapkan NER pada satu teks.
    Potong teks jika melebihi max_length karakter untuk efisiensi.
    Return string entitas yang ditemukan (deduplikasi).
    """
    cleaned = clean_text(text)[:max_length]
    results = ner_pipeline_id(cleaned, aggregation_strategy="simple")
    entities = list(set(f"{r['word']} ({r['entity_group']})" for r in results))
    return ', '.join(entities) if entities else '-'


def safe_perform_ner(text):
    """Wrapper dengan penanganan error."""
    try:
        return perform_ner(text)
    except Exception as e:
        return f"Error: {str(e)[:50]}"


print("Fungsi NER siap digunakan.")

Fungsi NER siap digunakan.


In [17]:
# Terapkan NER ke seluruh dataset
# Catatan: Proses ini mungkin memakan beberapa menit tergantung ukuran dataset
print("Memproses NER untuk seluruh dataset...")
df_text['NER_Results'] = df_text['text_final'].apply(safe_perform_ner)
print("NER selesai!")

# Tampilkan contoh hasil
print("\nContoh hasil NER:")
df_text[['text_final', 'NER_Results']].head(5)


Memproses NER untuk seluruh dataset...
NER selesai!

Contoh hasil NER:


,text_final,NER_Results
0,kali kartu kredit cocok anak muda update alam ...,-
1,hasil layan cs nama sangat sopan jelas sangat ...,##an cs (ORG)
2,cepat aman,-
3,cs baik,-
4,jelas baik,-


## 8) Simpan Hasil NER

Simpan DataFrame dengan hasil NER ke file CSV agar bisa digunakan untuk analisis lanjutan.

In [18]:
output_file = 'Honest_Reviews_NER.csv'
df_text.to_csv(output_file, index=False)
print(f"Hasil NER disimpan ke '{output_file}'")
print(f"Total ulasan diproses: {len(df_text):,}")

# Tampilkan semua data
df_text


Hasil NER disimpan ke 'Honest_Reviews_NER.csv'
Total ulasan diproses: 39,164


,score,content,text_final,reviewCreatedVersion,NER_Results
0,5,ini adalah pertama kali saya menggunakan kartu...,kali kartu kredit cocok anak muda update alam ...,3.831.0,-
1,5,"pengajuan berhasil, pelayanan utk CS atas nama...",hasil layan cs nama sangat sopan jelas sangat ...,3.833.1,##an cs (ORG)
2,5,Pengajuan cepat dan aman..,cepat aman,3.833.1,-
3,5,cs Ridwan terbaik,cs baik,3.833.1,-
4,5,penjelasan yang baik oleh mba rima,jelas baik,3.833.1,-
...,...,...,...,...,...
39159,5,Keren banget aplikasinya! Semua proses pengaju...,keren banget proses cepat tidak ribet tidak tu...,2.650.3,"the (EVT), next (PRD)"
39160,5,Beru kali ini nemu proses kartu kredit secepat...,kali nemu proses kartu kredit cepat menit lang...,NaN,-
39161,5,"Suka banget sama aplikasi ini, mudah mengajuka...",suka banget mudah via tidak ribet fleksible gu...,2.624.2,-
39162,5,Akhirnya ada juga aplikasi kartu kredit di Ind...,kartu kredit indonesia proses cepat mudah simp...,NaN,"kartu (ORG), kredit indonesia (PRD)"
